In [1]:
# Imports 
import torch
import torchvision
from torch import nn 
from torch import optim
from torchinfo import summary
import boto3 
import os 
import zipfile
from dataset import create_dataloaders
from train import train 

In [ ]:
# Load the data from s3 bucket 
BUCKET_NAME = "amzn-asl-s3-bucket"
FILE_KEY = "cnn-project/processed.zip"
LOCAL_PATH = "/home/ec2-user/SageMaker/processed.zip"
DATA_PATH = "/home/ec2-user/SageMaker/data/"

s3 = boto3.client("s3")
s3.download_file(BUCKET_NAME, FILE_KEY, LOCAL_PATH)

with zipfile.ZipFile(LOCAL_PATH, 'r') as zipref:
    zipref.extractall(DATA_PATH)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
DATA_PATH = "/home/ec2-user/SageMaker/data/"
train_path = DATA_PATH + "processed/train"
val_path = DATA_PATH + "processed/val"
test_path = DATA_PATH + "processed/test"
train_dataloader, val_dataloader, test_dataloader, train_data, val_data, _ = create_dataloaders(
        train_path, val_path, test_path
    )

In [4]:
# Load the CNN model and its weights 
weights = torchvision.models.ResNet18_Weights.DEFAULT
model = torchvision.models.resnet18(weights=weights).to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/ec2-user/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 250MB/s]


In [5]:
# Initialize the loss function and optimizer
loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [6]:
# Freeze the stem layer
for param in model.conv1.parameters():
    param.requires_grad = False
for param in model.bn1.parameters():
    param.requires_grad = False 

# Freeze layer 1 and layer 2 
for param in model.layer1.parameters():
    param.requires_grad = False 

model.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features=model.fc.in_features,
              out_features=len(train_data.classes))
).to(device)


In [7]:
# Display the CNN's summary
summary(model)

Layer (type:depth-idx)                   Param #
ResNet                                   --
├─Conv2d: 1-1                            (9,408)
├─BatchNorm2d: 1-2                       (128)
├─ReLU: 1-3                              --
├─MaxPool2d: 1-4                         --
├─Sequential: 1-5                        --
│    └─BasicBlock: 2-1                   --
│    │    └─Conv2d: 3-1                  (36,864)
│    │    └─BatchNorm2d: 3-2             (128)
│    │    └─ReLU: 3-3                    --
│    │    └─Conv2d: 3-4                  (36,864)
│    │    └─BatchNorm2d: 3-5             (128)
│    └─BasicBlock: 2-2                   --
│    │    └─Conv2d: 3-6                  (36,864)
│    │    └─BatchNorm2d: 3-7             (128)
│    │    └─ReLU: 3-8                    --
│    │    └─Conv2d: 3-9                  (36,864)
│    │    └─BatchNorm2d: 3-10            (128)
├─Sequential: 1-6                        --
│    └─BasicBlock: 2-3                   --
│    │    └─Conv2d: 3-11   

In [8]:
torch.manual_seed(42)
results = train(model, 0.001, train_dataloader, val_dataloader, 10, loss_func, optimizer, device)

Epoch 1
 Train Acc: 0.98, Train Loss: 0.08 | Validation Acc: 0.98, Validation Loss: 0.08
Epoch 2
 Train Acc: 0.99, Train Loss: 0.02 | Validation Acc: 1.00, Validation Loss: 0.00
Epoch 3
 Train Acc: 1.00, Train Loss: 0.01 | Validation Acc: 1.00, Validation Loss: 0.01
Epoch 4
 Train Acc: 1.00, Train Loss: 0.01 | Validation Acc: 1.00, Validation Loss: 0.00
Epoch 5
 Train Acc: 1.00, Train Loss: 0.01 | Validation Acc: 1.00, Validation Loss: 0.00
Epoch 6
 Train Acc: 1.00, Train Loss: 0.00 | Validation Acc: 1.00, Validation Loss: 0.00
Epoch 7
 Train Acc: 1.00, Train Loss: 0.01 | Validation Acc: 1.00, Validation Loss: 0.00
Epoch 8
 Train Acc: 1.00, Train Loss: 0.01 | Validation Acc: 1.00, Validation Loss: 0.00
Epoch 9
 Train Acc: 1.00, Train Loss: 0.00 | Validation Acc: 1.00, Validation Loss: 0.01
Epoch 10
 Train Acc: 1.00, Train Loss: 0.00 | Validation Acc: 1.00, Validation Loss: 0.00


In [9]:
# Save the model 
torch.save(model.state_dict(), 'model_v3.pth')